Sorry that I could not upload a file with the results. I ran it in Colab once but figured it would be a good add-on to include the loss function plot. I needed to run the code again, but I completely ran out of Colab runtime. Unfortunately, I can't run the code locally either, because my PC does not have a proper graphics card (Intel Iris). I hope that's alright, since the task itself mentions that it takes a lot of time."

In [ ]:
!pip install denoising_diffusion_pytorch

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.datasets as datasets
from torchvision import transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
from IPython.display import clear_output

from denoising_diffusion_pytorch import Unet, GaussianDiffusion

# 1. Device configuration (Crucial for speed)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Hyperparameters
LEARNING_RATE = 4e-4
BATCH_SIZE = 128
N_EPOCHS = 100
IMAGE_SIZE = 28
TIME_STEPS = 1000
SAMPLING_TIMESTEPS = 250

# Transform: Converts to Tensor (Values remain between 0 and 1)
myTransforms = transforms.Compose([transforms.ToTensor()])

# Load Dataset
print("Loading MNIST dataset...")
dataset = datasets.MNIST(root="dataset/", transform=myTransforms, download=True)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

test_dataset = datasets.MNIST(root='dataset/', train=False, download=False, transform=myTransforms)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

DIM = 32
DIM_MULTS = (1, 2, 5)

# Initialize U-Net
model = Unet(
    dim = DIM,
    dim_mults = DIM_MULTS,
    flash_attn = False,
    channels = 1
).to(device)

# Initialize Diffusion Model
diffusion = GaussianDiffusion(
    model,
    image_size = IMAGE_SIZE,
    timesteps = TIME_STEPS,
    sampling_timesteps = SAMPLING_TIMESTEPS
).to(device)

optim = optim.AdamW(model.parameters(), lr=LEARNING_RATE)

# List to store loss values for plotting
epoch_losses = []

# 2. Training Loop
print("Starting training...")
for epoch in range(N_EPOCHS):
    model.train()
    epoch_loss = 0.0

    for batch_idx, (images, _) in enumerate(loader):
        images = images.to(device)

        optim.zero_grad()
        loss = diffusion(images)
        loss.backward()
        optim.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(loader)
    epoch_losses.append(avg_loss) # Store the average loss for this epoch
    print(f"Epoch [{epoch+1}/{N_EPOCHS}], Loss: {avg_loss:.4f}")

    # 3. Sampling and plotting directly in the notebook
    if (epoch + 1) % 10 == 0 or epoch == N_EPOCHS - 1:
        model.eval()
        with torch.no_grad():
            print(f"Generating samples for epoch {epoch+1}...")
            # Sample 16 images
            sampled_images = diffusion.sample(batch_size = 16)

            grid = make_grid(sampled_images, nrow=4).permute(1, 2, 0).cpu().numpy()

            plt.figure(figsize=(5, 5))
            plt.imshow(grid)
            plt.title(f"Generated Digits - Epoch {epoch+1}")
            plt.axis('off')
            plt.show()

print("Training complete!")

Using device: cpu
Loading MNIST dataset...
Starting training...


In [ ]:
import matplotlib.pyplot as plt

# Plot the loss curve after training
plt.figure(figsize=(10, 6))
plt.plot(range(1, N_EPOCHS + 1), epoch_losses, marker='o', linestyle='-')
plt.title('Training Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Average Loss')
plt.grid(True)
plt.show()